# Hybrid Adaptive RAG-driven Generative AI

Copyright 2026, Denis Rothman

**The 2026 Perspective:**
By 2026, connecting to an LLM is a commodity task. The technical barrier to entry has collapsed. However, the *value* barrier has risen. Users no longer accept generic "AI-sounding" answers.

The core challenge today is **Domain Alignment** and **Nuance**. While models like **GPT-5.1** possess immense reasoning capabilities, they still lack the specific intent and unspoken context of human experts. This notebook demonstrates why the **Human-in-the-Loop (HITL)** architecture remains the critical differentiator between a functioning demo and a production-grade system.

- **Program Overview**:
  - **Retriever Phase**: Fetches raw domain data (Wikipedia). In 2026, this represents the "Ground Truth" layer.
  - **Generator Phase**: Uses **GPT-5.1** to synthesize answers. The focus here is on *steering* the model via dynamic context rather than just prompting.
  - **Evaluator Phase**: The critical layer. Uses cosine similarity for semantic checking, but relies on specific human feedback to capture "vibe," "tone," and "brand alignment"—metrics that automated evals still struggle with.

# 1. RETRIEVER

## 1.1. Installing the environment
Updating to the latest libraries compatible with the 2026 ecosystem.

In [ ]:
!pip install requests
!pip install beautifulsoup4==4.14.3
!pip install openai==2.26.0
!pip install scikit-learn==1.8.0

## 1.2. Preparing the dataset
Defining the dataset with labels and documents. We scrape live data to ensure our "Ground Truth" is current.

In [ ]:
import requests
from bs4 import BeautifulSoup
import re

# URLs of the Wikipedia articles mapped to keywords
urls = {
    "prompt engineering": "https://en.wikipedia.org/wiki/Prompt_engineering",
    "artificial intelligence":"https://en.wikipedia.org/wiki/Artificial_intelligence",
    "llm": "https://en.wikipedia.org/wiki/Large_language_model",
    "llms": "https://en.wikipedia.org/wiki/Large_language_model"
}

## 1.2.2. Processing the data

In [ ]:
def fetch_and_clean(url):
    # 1. Define headers to mirror a real browser for educational purposes and show we are not a bot(Essential for Wikipedia)
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    try:
        # 2. Fetch the URL with headers
        response = requests.get(url, headers=headers)

        # Check if the request was actually successful
        if response.status_code != 200:
            return f"Error: Server returned status code {response.status_code}. Request blocked."

        soup = BeautifulSoup(response.content, 'html.parser')

        # 3. Find the main content (Try standard class first, then fallback)
        content = soup.find('div', {'class': 'mw-parser-output'})

        if content is None:
            content = soup.find('div', {'id': 'bodyContent'})

        if content is None:
            return f"Error: Could not extract content from {url}. HTML structure mismatch."

        # 4. Remove sidebar/reference clutter
        for section_title in ['References', 'Bibliography', 'External links', 'See also']:
            section = content.find('span', {'id': section_title})
            if section:
                parent = section.find_parent()
                if parent:
                    for sib in parent.find_next_siblings():
                        sib.decompose()
                    parent.decompose()

        # 5. Extract text
        paragraphs = content.find_all('p')
        cleaned_text = ' '.join(paragraph.get_text(separator=' ', strip=True) for paragraph in paragraphs)
        cleaned_text = re.sub(r'\[\d+\]', '', cleaned_text) # Remove [1], [2] citations

        # Final check if text is empty
        if not cleaned_text:
            return "Error: Page was found but no text paragraphs could be extracted."

        return cleaned_text

    except Exception as e:
        return f"Error: An exception occurred: {str(e)}"

## 1.3. The retrieval process for user input

In [ ]:
import textwrap

def process_query(user_input, num_words):
    user_input = user_input.lower()

    # Check for any of the specified keywords in the input
    matched_keyword = next((keyword for keyword in urls if keyword in user_input), None)

    if matched_keyword:
        print(f"Fetching data from: {urls[matched_keyword]}")
        cleaned_text = fetch_and_clean(urls[matched_keyword])

        # Limit the display to the specified number of words from the cleaned text
        words = cleaned_text.split()  # Split the text into words
        first_n_words = ' '.join(words[:num_words])  # Join the first n words into a single string

        # Wrap the first n words to 80 characters wide for display
        wrapped_text = textwrap.fill(first_n_words, width=80)
        print("\nFirst {} words of the cleaned text:".format(num_words))
        print(wrapped_text)  # Print the first n words as a well-formatted paragraph

        # Use the exact same first_n_words for the Generator prompt to ensure consistency
        prompt = f"Summarize the following information about {matched_keyword}:\n{first_n_words}"
        wrapped_prompt = textwrap.fill(prompt, width=80)  # Wrap prompt text
        print("\nPrompt for Generator:", wrapped_prompt)

        # Return the specified number of words
        return first_n_words
    else:
        print("No relevant keywords found. Please enter a query related to 'LLM', 'LLMs', or 'Prompt Engineering'.")
        return None

In [ ]:
# --- Maintenance ---
# Uncomment it when adding new URLs to perform preliminary verifications if necessary.
'''
# We simulate a direct call to the function to see if it works.

# 1. Define a test input that definitely triggers a keyword (e.g., "llm")
test_input = "what is an llm?"
test_limit = 50

print(f"Testing 'process_query' with input: '{test_input}'\n" + "-"*40)

# 2. Call the function directly
# This skips the ranking logic and goes straight to retrieval
try:
    test_result = process_query(test_input, test_limit)

    if test_result:
        print("\n✅ SUCCESS: Content retrieved and processed:")
        print("-" * 20)
        print(test_result)
    else:
        print("\n⚠️ WARNING: The function ran but returned 'None'. Check your keywords.")

except Exception as e:
    print("\n❌ CRITICAL FAILURE: The function crashed.")
    print(f"Error details: {e}")
'''

# 2. GENERATOR

## 2.1. Adaptive RAG Selection
Integrating Human Feedback (HF) to augment document inputs.

In [ ]:
#@title download image
!curl -L "https://api.github.com/repos/Denis2054/RAG-Driven-Generative-AI-2nd-Edition/contents/Chapter08/rag_strategy.png" -o rag_strategy.png

In [ ]:
#@title show image
from IPython.display import Image, display

# Specify the path to your PNG file
image_path = '/content/rag_strategy.png'

# Display the image
display(Image(filename=image_path, width=650, height=400))

## 2.2. Input

In [ ]:
# Request user input for keyword parsing
user_input = input("Enter your query: ").lower()

## 2.3 Mean ranking simulation scenario

In 2026, "low ranking" usually doesn't mean the AI hallucinated wild facts; it means the AI was **generic** or **lacked domain nuance**.

We utilize a simulation variable `ranking` (1-5):
* **1-2:** The system provides raw data (No RAG).
* **3-4:** The system injects specific Human Feedback (HF) to correct tone/style.
* **5:** The system uses standard RAG (High confidence).

In [ ]:
# Select a score between 1 and 5 to run the simulation
ranking=5

In [ ]:
# Initializing the text for the generative AI model simulations
text_input=[]

### Ranking 1-2 : No RAG (Raw retrieval)

In [ ]:
if ranking>=1 and ranking<3:
  text_input=user_input

### Ranking 3-4 : Human Expert Feedback RAG only
Here we inject specific domain knowledge that GPT-5.1 might miss due to its generalization training.

In [ ]:
hf=False
if ranking>=3 and ranking<5:
  hf=True

Retrieve human expert feedback flashcards, documents or snippets.

In [ ]:
#@title Human Feedback option
if hf==True:
   !curl -L "https://api.github.com/repos/Denis2054/RAG-Driven-Generative-AI-2nd-Edition/contents/Chapter08/human_feedback.txt" -o human_feedback.txt

Check if human_feedback.txt exists and retrieve content.

In [ ]:
import os

if hf==True:
    efile = os.path.exists('human_feedback.txt')

    if efile:
        with open('human_feedback.txt', 'r') as file:
            content = file.read().replace('\n', ' ').replace('#', '')
        text_input=content
        print(text_input)
    else:
      print("File not found")
      hf=False

### Ranking 5 : RAG only with no HF
Standard operation using external knowledge bases.

In [ ]:
if ranking>=5:
  max_words=100 # Limit: the size of the data we can add to the input
  rdata=process_query(user_input,max_words)
  # print(rdata) # debug
  if rdata:
        rdata_clean = rdata.replace('\n', ' ').replace('#', '')
        rdata_sentences = rdata_clean.split('. ')
  text_input=rdata
  print(text_input)

## 2.4. Checking the input before running the generator

In [ ]:
print("user input:",user_input)

In [ ]:
print("text input:",text_input)

## 2.5. Installing the Generative AI environment
Note: Ensure you have a valid API key for GPT-5.1.

In [ ]:
import os
from openai import OpenAI
from google.colab import userdata

try:
    # Explicitly check for the mandatory OpenAI key
    openai_key = userdata.get("API_KEY")
    if not openai_key:
        raise ValueError("OpenAI API_KEY not found in Secrets.")

    os.environ["OPENAI_API_KEY"] = openai_key
    client = OpenAI()
    print("✅ OpenAI initialized (Mandatory).")

except Exception as e:
    print(f"❌ Critical Error: OpenAI initialization failed. {e}")
    # We raise the error here because OpenAI is required for the Planner to work at all
    raise e

## 2.6. Content generation
Using **GPT-5.1**. In 2026, prompt engineering focuses on "constraints" and "density" rather than trying to force reasoning, which the model now handles natively.

In [ ]:
from openai import OpenAI
import time

client = OpenAI()

# 2026 Update: We default to gpt-5.1 for production tasks.
gptmodel="gpt-5.1"
start_time = time.time()

def call_genai_with_full_text(itext):
    # Join all lines to form a single string
    text_input = '\n'.join(itext) if isinstance(itext, list) else itext
    prompt = f"Please summarize and refine the following content for a professional 2026 audience:\n{text_input}"

    try:
      response = client.chat.completions.create(
         model=gptmodel,
         messages=[
            {"role": "system", "content": "You are an advanced 2026 Domain Specialist. Address Users expectancy for high density, zero fluff, and perfect alignment with provided context."},
            {"role": "user", "content": prompt}
         ],
         temperature=0.1
        )
      return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error calling {gptmodel}: {str(e)}"

# Call the function and print the result
genai_response = call_genai_with_full_text(text_input)

response_time = time.time() - start_time
print(f"Response Time: {response_time:.2f} seconds")
print(gptmodel,"Response:", genai_response)

### Formatted response

In [ ]:
import textwrap

def print_formatted_response(response):
    wrapper = textwrap.TextWrapper(width=80)
    wrapped_text = wrapper.fill(text=response)
    print("AI Response:")
    print("---------------")
    print(wrapped_text)
    print("---------------\n")

print_formatted_response(genai_response)

# 3. EVALUATOR

## 3.1. Response time
Latency remains a key metric in 2026, especially for real-time agentic workflows.

In [ ]:
print(f"Response Time: {response_time:.2f} seconds")

## 3.2. Cosine Similarity Score
Measuring strict semantic adherence to the retrieved ground truth.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity(text1, text2):
    if not text1 or not text2:
        return 0.0
    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform([text1, text2])
    similarity = cosine_similarity(tfidf[0:1], tfidf[1:2])
    return similarity[0][0]

similarity_score = calculate_cosine_similarity(text_input, genai_response)
print(f"Cosine Similarity Score: {similarity_score:.3f}")

## 3.3. Human user rating
In 2026, specific criteria (Accuracy vs Vibe) are separated. Here we focus on **Relevance**.

In [ ]:
# Score parameters
counter=20                      # number of queries
score_history=60                # human feedback
threshold=4                     # minimum rankings to trigger human expert feedback

In [ ]:
import numpy as np

def evaluate_response(response):
    print("\nGenerated Response:")
    print(response)
    print("\nEvaluate based on 2026 Standards (Relevance, Density, Domain Fit):")
    print("1 - Poor, 2 - Fair, 3 - Good, 4 - Very Good, 5 - Excellent")
    score = input("Enter score (1-5): ")
    try:
        score = int(score)
        if 1 <= score <= 5:
            return score
        else:
            print("Invalid score.")
            return evaluate_response(response)
    except ValueError:
        print("Invalid input.")
        return evaluate_response(response)

score = evaluate_response(genai_response)
print("Evaluator Score:", score)

counter+=1
score_history+=score
mean_score=round(np.mean(score_history/counter), 2)
if counter>0:
  print("Rankings      :", counter)
  print("Score history : ", mean_score)

## 3.4. Human expert evaluation

**The 2026 Feedback Loop:**
Expert feedback is now captured via intuitive interfaces to create "Golden Datasets" for fine-tuning.

In [ ]:
#@title download images
!curl -L "https://api.github.com/repos/Denis2054/RAG-Driven-Generative-AI-2nd-Edition/contents/Chapter08/thumbs_up.png" -o thumbs_up.png
!curl -L "https://api.github.com/repos/Denis2054/RAG-Driven-Generative-AI-2nd-Edition/contents/Chapter08/thumbs_down.png" -o thumbs_down.png

In [ ]:
counter_threshold=10
score_threshold=4

if counter>counter_threshold and score_history<=score_threshold:
  print("Human expert evaluation is required for the feedback loop.")

Visual Interface for Expert Alignment:

In [ ]:
import base64
import uuid
from google.colab import output
from IPython.display import display, HTML

def image_to_data_uri(file_path):
    with open(file_path, 'rb') as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode()
        return f'data:image/png;base64,{encoded_string}'

# Ensure the paths point to where the images were downloaded
thumbs_up_data_uri = image_to_data_uri('/content/thumbs_up.png')
thumbs_down_data_uri = image_to_data_uri('/content/thumbs_down.png')

def display_icons():
    # Generate unique IDs for this execution to avoid DOM conflicts
    up_id = f"thumbs_up_{uuid.uuid4()}"
    down_id = f"thumbs_down_{uuid.uuid4()}"

    html = f'''
    <div style="font-family: sans-serif; border: 1px solid #ccc; padding: 10px; border-radius: 5px;">
      <p><b>Expert Feedback Interface (2026 Standard)</b></p>
      <p>Does this response align with domain nuance and brand voice?</p>
      <img src="{thumbs_up_data_uri}" id="{up_id}" style="cursor:pointer; width:50px; margin-right:20px;" />
      <img src="{thumbs_down_data_uri}" id="{down_id}" style="cursor:pointer; width:50px;" />
    </div>

    <script type="text/javascript">
        // Target specific unique IDs generated for this cell run
        document.querySelector("#{up_id}").onclick = function() {{
            google.colab.output.clear();
            google.colab.kernel.invokeFunction('notebook.save_feedback', ["POSITIVE ALIGNMENT"], {{}});
        }};
        document.querySelector("#{down_id}").onclick = function() {{
            const text = prompt("Please specify the domain nuance missed:");
            if (text !== null) {{
                google.colab.kernel.invokeFunction('notebook.save_feedback', [text], {{}});
            }}
        }};
    </script>
    '''
    display(HTML(html))

def save_feedback(feedback):
    with open('/content/expert_feedback.txt', 'w') as f:
        f.write(feedback)
    print("Feedback captured for fine-tuning loop.")
    if feedback != "POSITIVE ALIGNMENT":
        print(f"Nuance Note: {feedback}")

output.register_callback('notebook.save_feedback', save_feedback)
display_icons()

In [ ]:
import os
# Check if 'expert_feedback.txt' exists (verify loop closure)
if os.path.exists('expert_feedback.txt'):
    with open('expert_feedback.txt', 'r') as file:
      content = file.read()
      print("Stored Feedback:", content)